# Generate activity scores for the scRNA-seq benchmarks

This notebook generates the activity-score parquet files used by `mwu_delongs.ipynb`. It runs:

1. **z-aggregate** across the prior-network and weighting configurations used in the paper, and
2. **VIPER, ULM, and z-score** for the method comparison.

Datasets are discovered dynamically from `scRNASeq/*.h5ad`. The directory currently may contain only a subset of the benchmark datasets; any additional `.h5ad` files added later will be included automatically the next time the notebook is run. This notebook does not download datasets.


## Local project layout

The notebook can be launched from the repository root or from `reproduce/Reproduce scRNASeq Results/`.

- Input datasets: `scRNASeq/<dataset>.h5ad`
- Prior networks: repository-level `data/*.tsv`
- Generated scores: `scores/<dataset>/*.parquet`
- Local z-aggregate implementation: repository-level `src/`

The filenames written here match the patterns expected by the MWU and DeLong analysis scripts.


In [6]:
from pathlib import Path
import gc
import sys
import warnings

import decoupler as dc
import numpy as np
import pandas as pd
import scanpy as sc
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning, module="src.preprocessing")

current_dir = Path.cwd().resolve()
local_analysis_dir = current_dir / "reproduce" / "Reproduce scRNASeq Results"
analysis_dir = local_analysis_dir if local_analysis_dir.is_dir() else current_dir
repository_root = analysis_dir.parents[1]

if not (repository_root / "src").is_dir():
    raise FileNotFoundError(
        "Could not locate the repository-level src/ package. Start Jupyter from the "
        "repository root or from reproduce/Reproduce scRNASeq Results/."
    )

if str(repository_root) not in sys.path:
    sys.path.insert(0, str(repository_root))

from src.WeightType import WeightType
from src.preprocessing import (
    compute_network_weights,
    preprocess_adata,
    read_adata_file,
    read_prior_network_file,
)
from src.z_aggregate import run_z_aggregate

dataset_directory = analysis_dir / "scRNASeq"
scores_directory = analysis_dir / "scores"
prior_directory = repository_root / "data"

scores_directory.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {repository_root}")
print(f"Dataset directory: {dataset_directory}")
print(f"Scores directory: {scores_directory}")


Repository root: /Users/kisanthapa/Downloads/z-aggregate
Dataset directory: /Users/kisanthapa/Downloads/z-aggregate/reproduce/Reproduce scRNASeq Results/scRNASeq
Scores directory: /Users/kisanthapa/Downloads/z-aggregate/reproduce/Reproduce scRNASeq Results/scores


## Discover datasets

Every `.h5ad` file in `scRNASeq/` is treated as one dataset. Because discovery happens at execution time, no notebook changes are needed when the remaining datasets are added.


In [7]:
dataset_paths = sorted(dataset_directory.glob("*.h5ad"))

if not dataset_paths:
    raise FileNotFoundError(f"No .h5ad datasets found in {dataset_directory}")

datasets = [path.stem for path in dataset_paths]

pd.DataFrame(
    {
        "dataset": datasets,
        "path": [str(path) for path in dataset_paths],
    }
)


,dataset,path
0,PapalexiSatija2021_eccite_RNA,/Users/kisanthapa/Downloads/z-aggregate/reprod...
1,TianKampmann2021_CRISPRa,/Users/kisanthapa/Downloads/z-aggregate/reprod...


## Analysis configuration

The configuration below reproduces the score combinations consumed by the downstream comparisons:

- z-aggregate with four prior networks using uniform weights,
- z-aggregate with four weighting strategies using the CausalPath prior, and
- VIPER, ULM, and z-score using the CausalPath prior with uniform weights.

`overwrite = False` keeps existing parquet files and computes only missing outputs.


In [8]:
min_targets = 5
min_genes = 1000
min_cells = 10
max_mt_pct = 20.0
decoupler_batch_size = 10_000
overwrite = False

prior_paths = {
    "causalpath-priors": prior_directory / "causalpath-priors.tsv",
    "collectri": prior_directory / "collectri.tsv",
    "dorothea": prior_directory / "dorothea.tsv",
    "ensemble-priors": prior_directory / "ensemble-priors.tsv",
}

weight_types = {
    "UNIFORM": WeightType.UNIFORM,
    "CORRELATION": WeightType.CORRELATION,
    "SPECIFICITY": WeightType.SPECIFICITY,
    "NONZERORATE": WeightType.NON_ZERO_RATE,
}

z_aggregate_configurations = [
    (prior_name, "UNIFORM") for prior_name in prior_paths
] + [
    ("causalpath-priors", weight_name)
    for weight_name in ("CORRELATION", "SPECIFICITY", "NONZERORATE")
]

comparison_methods = ["viper", "ulm", "zscore"]

missing_priors = [str(path) for path in prior_paths.values() if not path.is_file()]
if missing_priors:
    raise FileNotFoundError(f"Missing prior-network files: {missing_priors}")

z_aggregate_configurations


[('causalpath-priors', 'UNIFORM'),
 ('collectri', 'UNIFORM'),
 ('dorothea', 'UNIFORM'),
 ('ensemble-priors', 'UNIFORM'),
 ('causalpath-priors', 'CORRELATION'),
 ('causalpath-priors', 'SPECIFICITY'),
 ('causalpath-priors', 'NONZERORATE')]

## Run z-aggregate from the local `src` package

This section imports and calls the repository implementation directly:

- `src.preprocessing.read_adata_file` loads each AnnData file.
- `src.preprocessing.preprocess_adata` performs cell/gene filtering, mitochondrial filtering, library-size normalization, and `log1p` transformation.
- `src.preprocessing.compute_network_weights` applies the selected weighting strategy.
- `src.z_aggregate.run_z_aggregate` computes TF activity scores and p-values.

Only the activity scores are saved because those are the inputs required by the MWU and DeLong analyses. Output files follow:

`<dataset>_z-agg_<prior>_<WEIGHT>.parquet`


In [9]:
def load_preprocessed_dataset(dataset_path: Path):
    adata = read_adata_file(str(dataset_path))
    return preprocess_adata(
        adata,
        min_genes=min_genes,
        min_cells=min_cells,
        max_mt_pct=max_mt_pct,
    )


def load_prior_networks() -> dict[str, pd.DataFrame]:
    return {
        prior_name: read_prior_network_file(str(prior_path))
        for prior_name, prior_path in prior_paths.items()
    }


def prepare_network(
    adata,
    prior_network: pd.DataFrame,
    weight_type: WeightType,
) -> pd.DataFrame:
    network = prior_network[
        prior_network["target"].isin(set(adata.var_names))
    ].copy()

    target_counts = network.groupby("source")["target"].nunique()
    valid_sources = target_counts[target_counts >= min_targets].index
    network = network[network["source"].isin(valid_sources)].reset_index(drop=True)

    if network.empty:
        raise ValueError("No regulator-target pairs remain after dataset filtering.")

    return compute_network_weights(
        adata=adata,
        prior_network=network,
        weight_type=weight_type,
    )


prior_networks = load_prior_networks()
{name: len(network) for name, network in prior_networks.items()}


{'causalpath-priors': 12071,
 'collectri': 42871,
 'dorothea': 32286,
 'ensemble-priors': 75341}

In [10]:
def run_z_aggregate_for_dataset(dataset_path: Path) -> list[dict]:
    dataset_name = dataset_path.stem
    output_directory = scores_directory / dataset_name
    output_directory.mkdir(parents=True, exist_ok=True)

    pending = []
    for prior_name, weight_name in z_aggregate_configurations:
        output_path = (
            output_directory
            / f"{dataset_name}_z-agg_{prior_name}_{weight_name}.parquet"
        )
        if overwrite or not output_path.exists():
            pending.append((prior_name, weight_name, output_path))

    if not pending:
        print(f"Skipping z-aggregate for {dataset_name}: all outputs exist")
        return []

    print(f"Loading and preprocessing {dataset_name}")
    adata = load_preprocessed_dataset(dataset_path)
    rows = []

    for prior_name, weight_name, output_path in pending:
        print(f"  z-aggregate: prior={prior_name}, weight={weight_name}")
        network = prepare_network(
            adata=adata,
            prior_network=prior_networks[prior_name],
            weight_type=weight_types[weight_name],
        )

        scores, pvalues = run_z_aggregate(
            adata=adata,
            priors=network,
            min_targets=min_targets,
        )
        scores = scores.sort_index(axis=1)
        scores.to_parquet(output_path, engine="pyarrow")

        rows.append(
            {
                "dataset": dataset_name,
                "method": "z-agg",
                "prior": prior_name,
                "weight": weight_name,
                "cells": scores.shape[0],
                "regulators": scores.shape[1],
                "output": str(output_path),
            }
        )

        del network, scores, pvalues
        gc.collect()

    del adata
    gc.collect()
    return rows


In [11]:
z_aggregate_summary = []

for dataset_path in tqdm(dataset_paths, desc="Datasets: z-aggregate"):
    z_aggregate_summary.extend(run_z_aggregate_for_dataset(dataset_path))

pd.DataFrame(z_aggregate_summary)


Datasets: z-aggregate:   0%|          | 0/2 [00:00<?, ?it/s]

2026-06-14 15:20:24 | [INFO] Reading expression data from: /Users/kisanthapa/Downloads/z-aggregate/reproduce/Reproduce scRNASeq Results/scRNASeq/TianKampmann2021_CRISPRa.h5ad


Skipping z-aggregate for PapalexiSatija2021_eccite_RNA: all outputs exist
Loading and preprocessing TianKampmann2021_CRISPRa


2026-06-14 15:20:26 | [INFO]    Loaded data shape: 21193 cells x 33538 genes
2026-06-14 15:20:26 | [INFO] Starting Preprocessing...
2026-06-14 15:20:27 | [INFO]    Filtered cells (min_genes=1000): Removed 941 cells.
2026-06-14 15:20:27 | [INFO]    Filtered genes (min_cells=10): Removed 13170 genes.
2026-06-14 15:20:28 | [INFO]    Mitochondrial filter (<20.0%): Removed 1097 cells.
2026-06-14 15:20:29 | [INFO] Preprocessing complete. Final shape: 19155 cells x 20368 genes
2026-06-14 15:20:29 | [INFO] Computing weights using strategy: Uniform
2026-06-14 15:20:29 | [INFO]    Uniform weights: using interaction as weight (no overlap filtering).
2026-06-14 15:20:29 | [INFO]    Weights computed successfully.


  z-aggregate: prior=causalpath-priors, weight=UNIFORM


...

  z-aggregate: prior=causalpath-priors, weight=SPECIFICITY


2026-06-14 15:22:29 | [INFO] Computing weights using strategy: NonzeroRate
2026-06-14 15:22:29 | [INFO]    Network Overlap: 7484/7484 edges (100.00%) target genes present in dataset.
2026-06-14 15:22:29 | [INFO]    Calculating nonzero rate weights...


  z-aggregate: prior=causalpath-priors, weight=NONZERORATE


2026-06-14 15:22:29 | [INFO]    Weights computed successfully.


,dataset,method,prior,weight,cells,regulators,output
0,TianKampmann2021_CRISPRa,z-agg,causalpath-priors,UNIFORM,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...
1,TianKampmann2021_CRISPRa,z-agg,collectri,UNIFORM,19155,707,/Users/kisanthapa/Downloads/z-aggregate/reprod...
2,TianKampmann2021_CRISPRa,z-agg,dorothea,UNIFORM,19155,290,/Users/kisanthapa/Downloads/z-aggregate/reprod...
3,TianKampmann2021_CRISPRa,z-agg,ensemble-priors,UNIFORM,19155,948,/Users/kisanthapa/Downloads/z-aggregate/reprod...
4,TianKampmann2021_CRISPRa,z-agg,causalpath-priors,CORRELATION,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...
5,TianKampmann2021_CRISPRa,z-agg,causalpath-priors,SPECIFICITY,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...
6,TianKampmann2021_CRISPRa,z-agg,causalpath-priors,NONZERORATE,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...


## Run VIPER, ULM, and z-score with Decoupler

For the method comparison, each dataset is preprocessed and then scaled before running the three Decoupler methods together. The signed CausalPath network uses uniform weights. Outputs follow:

`<dataset>_<method>_causalpath-priors_UNIFORM.parquet`

These names match the method-comparison loader in `mwu_delongs_scripts/Methods_MWU-Delongs.py`.


In [12]:
def run_comparison_methods_for_dataset(dataset_path: Path) -> list[dict]:
    dataset_name = dataset_path.stem
    output_directory = scores_directory / dataset_name
    output_directory.mkdir(parents=True, exist_ok=True)

    output_paths = {
        method: output_directory
        / f"{dataset_name}_{method}_causalpath-priors_UNIFORM.parquet"
        for method in comparison_methods
    }
    pending_methods = [
        method
        for method, output_path in output_paths.items()
        if overwrite or not output_path.exists()
    ]

    if not pending_methods:
        print(f"Skipping comparison methods for {dataset_name}: all outputs exist")
        return []

    print(f"Loading and preprocessing {dataset_name}")
    adata = load_preprocessed_dataset(dataset_path)
    network = prepare_network(
        adata=adata,
        prior_network=prior_networks["causalpath-priors"],
        weight_type=WeightType.UNIFORM,
    )

    sc.pp.scale(adata)
    dc.mt.decouple(
        adata,
        network,
        methods=comparison_methods,
        tmin=min_targets,
        bsize=decoupler_batch_size,
        verbose=True,
    )

    rows = []
    for method in pending_methods:
        result_key = f"score_{method}"
        if result_key not in adata.obsm:
            raise KeyError(f"Decoupler did not create adata.obsm[{result_key!r}]")

        scores = adata.obsm[result_key].astype(np.float64).sort_index(axis=1)
        scores.to_parquet(output_paths[method], engine="pyarrow")
        rows.append(
            {
                "dataset": dataset_name,
                "method": method,
                "prior": "causalpath-priors",
                "weight": "UNIFORM",
                "cells": scores.shape[0],
                "regulators": scores.shape[1],
                "output": str(output_paths[method]),
            }
        )

    del adata, network
    gc.collect()
    return rows


In [13]:
comparison_summary = []

for dataset_path in tqdm(dataset_paths, desc="Datasets: comparison methods"):
    comparison_summary.extend(run_comparison_methods_for_dataset(dataset_path))

pd.DataFrame(comparison_summary)


Datasets: comparison methods:   0%|          | 0/2 [00:00<?, ?it/s]

2026-06-14 15:23:02 | [INFO] Reading expression data from: /Users/kisanthapa/Downloads/z-aggregate/reproduce/Reproduce scRNASeq Results/scRNASeq/TianKampmann2021_CRISPRa.h5ad


Skipping comparison methods for PapalexiSatija2021_eccite_RNA: all outputs exist
Loading and preprocessing TianKampmann2021_CRISPRa


2026-06-14 15:23:04 | [INFO]    Loaded data shape: 21193 cells x 33538 genes
2026-06-14 15:23:04 | [INFO] Starting Preprocessing...
2026-06-14 15:23:05 | [INFO]    Filtered cells (min_genes=1000): Removed 941 cells.
2026-06-14 15:23:05 | [INFO]    Filtered genes (min_cells=10): Removed 13170 genes.
2026-06-14 15:23:06 | [INFO]    Mitochondrial filter (<20.0%): Removed 1097 cells.
2026-06-14 15:23:07 | [INFO] Preprocessing complete. Final shape: 19155 cells x 20368 genes
2026-06-14 15:23:07 | [INFO] Computing weights using strategy: Uniform
2026-06-14 15:23:07 | [INFO]    Uniform weights: using interaction as weight (no overlap filtering).
2026-06-14 15:23:07 | [INFO]    Weights computed successfully.
2026-06-14 15:23:10 | [INFO] ulm - Running ulm
2026-06-14 15:23:12 | [INFO] Extracted omics mat with 19155 rows (observations) and 20368 columns (features)
2026-06-14 15:23:23 | [INFO] Network adjacency matrix has 1870 unique features and 444 unique sources
2026-06-14 15:23:23 | [INFO] ulm

  0%|          | 0/19155 [00:00<?, ?it/s]

2026-06-14 15:30:25 | [INFO] viper - adjusting p-values by FDR
2026-06-14 15:30:25 | [INFO] viper - done


,dataset,method,prior,weight,cells,regulators,output
0,TianKampmann2021_CRISPRa,viper,causalpath-priors,UNIFORM,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...
1,TianKampmann2021_CRISPRa,ulm,causalpath-priors,UNIFORM,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...
2,TianKampmann2021_CRISPRa,zscore,causalpath-priors,UNIFORM,19155,444,/Users/kisanthapa/Downloads/z-aggregate/reprod...


## Next step

After all expected datasets and score files are present, run `mwu_delongs.ipynb` to compute the MWU and top-two DeLong results. Re-running this notebook with `overwrite = False` skips score files that already exist.
